In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA_DIR = Path("../data/raw/samanantar_ml")

files = sorted(DATA_DIR.glob("*.parquet"))

print("Number of files:", len(files))

for file in files:
    print(file.name)

Number of files: 4
train-00000-of-00004.parquet
train-00001-of-00004.parquet
train-00002-of-00004.parquet
train-00003-of-00004.parquet


In [3]:
df = pd.read_parquet(files[0])

print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 1481107
Columns: ['idx', 'src', 'tgt']


In [4]:
df.head()

,idx,src,tgt
0,0,The plot of the movie revolves around the life...,"ക്യാന്‍സറിനോട് പോരാടുന്ന കിസി, മാനി എന്നിവരുടെ..."
1,1,The same as on the stone.,കല്ലിൽ കണ്ട അതേ രൂപം.
2,2,"Oommen Chandy, Mullappally Ramachandran, Rames...",മാണി നയിക്കുന്ന കേരള യാത്രയിലുടനീളം ഉമ്മന്‍ചാണ...
3,3,Many were arrested and jailed.,നിരപരാധികളായ പലരേയും കസ്റ്റഡിയിലെടുത്ത് പീഢിപ്...
4,4,what has happened?,എന്താണു സംഭവം?


In [5]:
print("Dataset Shape:", df.shape)
print("\nColumn Information:")
print(df.info())

Dataset Shape: (1481107, 3)

Column Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1481107 entries, 0 to 1481106
Data columns (total 3 columns):
 #   Column  Non-Null Count    Dtype 
---  ------  --------------    ----- 
 0   idx     1481107 non-null  int64 
 1   src     1481107 non-null  object
 2   tgt     1481107 non-null  object
dtypes: int64(1), object(2)
memory usage: 33.9+ MB
None


In [6]:
print("\nMissing Values:")
print(df.isnull().sum())


Missing Values:
idx    0
src    0
tgt    0
dtype: int64


In [7]:
print("\nDuplicate Rows:", df.duplicated().sum())

print("Duplicate English Sentences:", df["src"].duplicated().sum())
print("Duplicate Malayalam Sentences:", df["tgt"].duplicated().sum())


Duplicate Rows: 0
Duplicate English Sentences: 442629
Duplicate Malayalam Sentences: 44865


In [8]:
print("\nDuplicate Rows:", df.duplicated().sum())

print("Duplicate English Sentences:", df["src"].duplicated().sum())
print("Duplicate Malayalam Sentences:", df["tgt"].duplicated().sum())


Duplicate Rows: 0
Duplicate English Sentences: 442629
Duplicate Malayalam Sentences: 44865


In [9]:
for i in range(5):
    print(f"\n--- Example {i+1} ---")
    print("English   :", df.loc[i, "src"])
    print("Malayalam :", df.loc[i, "tgt"])


--- Example 1 ---
English   : The plot of the movie revolves around the life of two cancer patients Kizie and Manny.
Malayalam : ക്യാന്‍സറിനോട് പോരാടുന്ന കിസി, മാനി എന്നിവരുടെ ജീവിതമാണ് ചിത്രം പറയുന്നത്.

--- Example 2 ---
English   : The same as on the stone.
Malayalam : കല്ലിൽ കണ്ട അതേ രൂപം.

--- Example 3 ---
English   : Oommen Chandy, Mullappally Ramachandran, Ramesh Chennithala and P.K.
Malayalam : മാണി നയിക്കുന്ന കേരള യാത്രയിലുടനീളം ഉമ്മന്‍ചാണ്ടി, പി. കെ കുഞ്ഞാലിക്കുട്ടി, മുല്ലപ്പള്ളി രാമചന്ദ്രന്‍, രമേശ് ചെന്നിത്തല, എം.

--- Example 4 ---
English   : Many were arrested and jailed.
Malayalam : നിരപരാധികളായ പലരേയും കസ്റ്റഡിയിലെടുത്ത് പീഢിപ്പിക്കുകയുമാണ്.

--- Example 5 ---
English   : what has happened?
Malayalam : എന്താണു സംഭവം?


In [10]:
print("Rows:", len(df))
print("Columns:", df.columns.tolist())

Rows: 1481107
Columns: ['idx', 'src', 'tgt']


In [11]:
df["src_words"] = df["src"].str.split().str.len()
df["tgt_words"] = df["tgt"].str.split().str.len()

print("English word count:")
print(df["src_words"].describe())

print("\nMalayalam word count:")
print(df["tgt_words"].describe())

English word count:
count    1.481107e+06
mean     1.020258e+01
std      9.083081e+00
min      1.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.300000e+01
max      3.149000e+03
Name: src_words, dtype: float64

Malayalam word count:
count    1.481107e+06
mean     6.961649e+00
std      5.867122e+00
min      1.000000e+00
25%      4.000000e+00
50%      5.000000e+00
75%      8.000000e+00
max      8.290000e+02
Name: tgt_words, dtype: float64


In [12]:
print("English sentences with 1 word:")
print((df["src_words"] == 1).sum())

print("Malayalam sentences with 1 word:")
print((df["tgt_words"] == 1).sum())

print("English sentences > 100 words:")
print((df["src_words"] > 100).sum())

print("Malayalam sentences > 100 words:")
print((df["tgt_words"] > 100).sum())

English sentences with 1 word:
964
Malayalam sentences with 1 word:
12082
English sentences > 100 words:
446
Malayalam sentences > 100 words:
78


In [13]:
translation_counts = df.groupby("src")["tgt"].nunique()

print("English sentences with multiple Malayalam translations:",
      (translation_counts > 1).sum())

print("\nMaximum different Malayalam translations for one English sentence:",
      translation_counts.max())

English sentences with multiple Malayalam translations: 197366

Maximum different Malayalam translations for one English sentence: 199


In [14]:
repeated_src = df[df["src"].duplicated(keep=False)].sort_values("src")

repeated_src[["src", "tgt"]].head(20)

,src,tgt
315254,! I gasped.,ഞാനലറി.
878676,! I gasped.,’ ഞാൻ കിതച്ചു.
223371,! I said with a shiver.,’’ ഞാൻ നീരസത്തോടെ പറഞ്ഞു.
755617,! I said with a shiver.,വിറയലോടെ ഞാന്‍ പറഞ്ഞു.
1477456,! he said.,' അദ്ദേഹം ഓര്‍ത്തു.
634469,! he said.,’ അയാളോർത്തു.
1312743,! she asked!,’ എന്ന് ചോദിക്കുന്നു!
1306779,! she asked!,""""""" അവർ ചോദിച്ചു !"""
496707,! she asked!,’ എന്നു ചോദിച്ചു!
614708,! she asked!,’ അവർ ചോദിക്കുന്നു!


In [15]:
samples = df.sample(20, random_state=42)

for i, row in samples.iterrows():
    print("\n--------------------------------")
    print("EN:", row["src"])
    print("ML:", row["tgt"])


--------------------------------
EN: I'm not taking my eyes off the road.
ML: ഞാൻ റോഡിൽ നിന്ന് കണ്ണുകൾ എടുക്കുന്നേയില്ല.

--------------------------------
EN: Various reforms have been undertaken since independence
ML: സ്വാതന്ത്ര്യലബ്ധിക്കുശേഷം വിവിധ പരിഷ്‌കാരങ്ങള്‍ നടത്തിയിട്ടുണ്ട്.

--------------------------------
EN: Recall the many touching and emotional scenes we read about in the Bible.
ML: ബൈബിളിൽ നാം വായിച്ചിട്ടുള്ള വികാരനിർഭരമായ ചില സന്ദർഭങ്ങളെക്കുറിച്ച് ഓർക്കുക.

--------------------------------
EN: The death toll has risen to 71 there.
ML: ഇതോടെ മരിച്ചവരുടെ എണ്ണം 71 ആയി.

--------------------------------
EN: There might be some problems in the personal relationship.
ML: വ്യക്തിപരമായ ബന്ധങ്ങളില്‍ ചില വിട്ടുവീഴ്ചകള്‍ വേണ്ടി വന്നേക്കാം.

--------------------------------
EN: The protesters who were students have been demanding justice for the victim.
ML: കൊല്ലപ്പെട്ട വിദ്യാര്‍ഥികള്‍ക്ക് നീതി ലഭ്യമാക്കണമെന്ന് പ്രതിഷേധക്കാര്‍ ആവശ്യപ്പെട്ടു.

--------------------------------
EN: 

In [16]:
import re

def malayalam_char_ratio(text):
    if not isinstance(text, str) or len(text) == 0:
        return 0
    
    mal_chars = re.findall(r'[\u0D00-\u0D7F]', text)
    letters = re.findall(r'[^\W\d_]', text, flags=re.UNICODE)
    
    if len(letters) == 0:
        return 0
    
    return len(mal_chars) / len(letters)


df["ml_script_ratio"] = df["tgt"].apply(malayalam_char_ratio)

print(df["ml_script_ratio"].describe())

count    1.481107e+06
mean     1.719067e+00
std      1.230977e-01
min      1.837838e-01
25%      1.661765e+00
50%      1.725490e+00
75%      1.787879e+00
max      4.000000e+00
Name: ml_script_ratio, dtype: float64


In [17]:
print("Pairs with no Malayalam script:",
      (df["ml_script_ratio"] == 0).sum())

print("Pairs with less than 50% Malayalam script:",
      (df["ml_script_ratio"] < 0.5).sum())

print("Pairs with more than 90% Malayalam script:",
      (df["ml_script_ratio"] > 0.9).sum())

Pairs with no Malayalam script: 0
Pairs with less than 50% Malayalam script: 660
Pairs with more than 90% Malayalam script: 1478048


In [18]:
df["length_ratio"] = (
    df["tgt_words"] / df["src_words"]
)

print(df["length_ratio"].describe())

count    1.481107e+06
mean     7.603162e-01
std      3.634575e-01
min      1.183432e-02
25%      5.625000e-01
50%      7.000000e-01
75%      9.000000e-01
max      1.658000e+02
Name: length_ratio, dtype: float64


In [19]:
print("Very short Malayalam relative to English:",
      (df["length_ratio"] < 0.2).sum())

print("Very long Malayalam relative to English:",
      (df["length_ratio"] > 3).sum())

Very short Malayalam relative to English: 5757
Very long Malayalam relative to English: 1678


In [20]:
suspicious = df[
    (df["ml_script_ratio"] < 0.5)
].copy()

print("Suspicious pairs:", len(suspicious))

for i, row in suspicious.head(20).iterrows():
    print("\n-------------------------")
    print("EN:", row["src"])
    print("ML:", row["tgt"])

Suspicious pairs: 660

-------------------------
EN: Redwood City
ML: സെഡാര്‍ സിറ്റിCity in California USA

-------------------------
EN: In R.R. Wise. J.K. Hoober.
ML: എന്നതിൽ R.R. Wise. J.K. Hoober.

-------------------------
EN: No power in the world can ever seperate Kashmir from India.
ML: ലോകത്തില്‍ ഒരു ശക്തിക്കും കശ്മീരിനെ ഇന്ത്യയില്‍ നിന്നും കവരാനാകില്ല,no one can take Kashmir from India says rajnath singhkeralaonlinenews.com | Malayalam news, kerala news, onlinenews,

-------------------------
EN: """Mohammed renames Emirates Road after Sheikh Mohammed bin Zayed""."
ML: "വിക്കിവൊയേജിൽ നിന്നുള്ള Emirates Road യാത്രാ സഹായി ""Mohammed renames Emirates Road after Sheikh Mohammed bin Zayed""."

-------------------------
EN: Third quarter
ML: മൂന്നാം കാല്‍ഭാഗംmoon phase between new moon and 1st quarter

-------------------------
EN: Intervene more aggressively to soften Rupee drop, government tells RBI: Report
ML: രൂപയുടെ വീഴ്ച തടയാന്‍ ശക്തമായി ഇടപെടണം. ആര്‍ബിഐക്ക് സര്‍ക്കാര്‍ നിര്‍

In [21]:
extreme = df[
    (df["src_words"] > 100) |
    (df["tgt_words"] > 100)
]

print("Extreme-length pairs:", len(extreme))

Extreme-length pairs: 498


In [22]:
for i, row in extreme.head(10).iterrows():
    print("\n-------------------------")
    print("English words:", row["src_words"])
    print("Malayalam words:", row["tgt_words"])
    print("EN:", row["src"][:500])
    print("ML:", row["tgt"][:500])


-------------------------
English words: 111
Malayalam words: 68
EN: "When Talut set forth with the armies, he said: ""Allah will test you at the stream: if any drinks of its water, He goes not with my army: Only those who taste not of it go with me: A mere sip out of the hand is excused."" but they all drank of it, except a few. When they crossed the river,- He and the faithful ones with him,- they said: ""This day We cannot cope with Goliath and his forces."" but those who were convinced that they must meet Allah, said: ""How oft, by Allah's will, Hath a small 
ML: "അങ്ങനെ പട്ടാളവുമായി ത്വാലൂത്ത് പുറപ്പെട്ടപ്പോള്‍ ‎പറഞ്ഞു: “അല്ലാഹു ഒരു നദികൊണ്ട് നിങ്ങളെ ‎പരീക്ഷിക്കാന്‍ പോവുകയാണ്. അതില്‍നിന്ന് ‎കുടിക്കുന്നവനാരോ, അവന്‍ എന്റെ കൂട്ടത്തില്‍ ‎പെട്ടവനല്ല. അത് രുചിച്ചുനോക്കാത്തവനാരോ അവനാണ് ‎എന്റെ അനുയായി. എന്നാല്‍ തന്റെ കൈകൊണ്ട് ഒരു ‎കോരല്‍ എടുത്തവന്‍ ഇതില്‍ നിന്നൊഴിവാണ്."" പക്ഷേ, ‎അവരില്‍ ചുരുക്കം ചിലരൊഴികെ എല്ലാവരും ‎അതില്‍നിന്ന് ഇഷ്ടംപോലെ കുടിച്ചു. അങ്ങനെ ‎ത്വാലൂത്തും കൂടെയുള്ള വിശ്വാസിക

In [23]:
import re

def script_stats(text):
    if not isinstance(text, str) or not text.strip():
        return pd.Series([0, 0, 0])

    malayalam = len(re.findall(r'[\u0D00-\u0D7F]', text))
    latin = len(re.findall(r'[A-Za-z]', text))
    digits = len(re.findall(r'\d', text))

    return pd.Series([malayalam, latin, digits])


df[["malayalam_chars", "latin_chars", "digit_chars"]] = (
    df["tgt"].apply(script_stats)
)

In [24]:
df["total_relevant_chars"] = (
    df["malayalam_chars"] +
    df["latin_chars"]
)

df["latin_ratio"] = (
    df["latin_chars"] /
    df["total_relevant_chars"].replace(0, np.nan)
)

print(df["latin_ratio"].describe())

count    1.481107e+06
mean     2.227061e-03
std      2.707039e-02
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      8.308458e-01
Name: latin_ratio, dtype: float64


In [25]:
print(
    "Pairs with >20% Latin characters:",
    (df["latin_ratio"] > 0.20).sum()
)

print(
    "Pairs with >50% Latin characters:",
    (df["latin_ratio"] > 0.50).sum()
)

Pairs with >20% Latin characters: 5560
Pairs with >50% Latin characters: 1339


In [26]:
html_pattern = r"<[^>]+>|https?://|www\.|\.com\b|\.in\b"

df["possible_web_noise"] = df["tgt"].str.contains(
    html_pattern,
    regex=True,
    case=False,
    na=False
)

print(
    "Possible web-noise pairs:",
    df["possible_web_noise"].sum()
)

Possible web-noise pairs: 354


In [27]:
web_noise = df[df["possible_web_noise"]]

for _, row in web_noise.head(20).iterrows():
    print("\n-------------------------")
    print("EN:", row["src"])
    print("ML:", row["tgt"])


-------------------------
EN: Water level in Idukki dam rises, reservoir level at highest
ML: ഇടുക്കി ഡാമിൽ ജലനിരപ്പ് ഉയരുന്നു. ചെറുതോണി, മാട്ടുപ്പെട്ടി ഡാമിൽ നിന്നും കൂടുതൽ വെള്ളം തുറന്നുവിടും | Twentyfournews.com

-------------------------
EN: No power in the world can ever seperate Kashmir from India.
ML: ലോകത്തില്‍ ഒരു ശക്തിക്കും കശ്മീരിനെ ഇന്ത്യയില്‍ നിന്നും കവരാനാകില്ല,no one can take Kashmir from India says rajnath singhkeralaonlinenews.com | Malayalam news, kerala news, onlinenews,

-------------------------
EN: Intervene more aggressively to soften Rupee drop, government tells RBI: Report
ML: രൂപയുടെ വീഴ്ച തടയാന്‍ ശക്തമായി ഇടപെടണം. ആര്‍ബിഐക്ക് സര്‍ക്കാര്‍ നിര്‍ദ്ദേശം,intervene more aggressively to soften rupee drop government tells rbikeralaonlinenews.com | Malayalam news, kerala news, onlinenews,

-------------------------
EN: Additionally, India has demonstrated how it rises up to challenges and uncovers opportunities therein, as manifested in the re-purposing of various au

In [28]:
short_pairs = df.sort_values(
    "length_ratio"
).head(20)

for _, row in short_pairs.iterrows():
    print("\n-------------------------")
    print("EN:", row["src"])
    print("ML:", row["tgt"])
    print("Ratio:", row["length_ratio"])


-------------------------
EN: These are all the text jobs. The State column may be: Queued - the job is waiting and will not be spoken until its state is changed to Waiting by clicking the Resume or Restart buttons. Waiting - the job is ready to be spoken. It will be spoken when the jobs preceding it in the list have finished. Speaking - the job is speaking. The Position column shows the current sentence of the job being spoken. You may pause a speaking job by clicking the Hold button. Paused - the job is currently paused. Paused jobs prevent jobs below them from speaking. Use the Resume or Restart buttons to resume speaking the job, or click Later to move the job down in the list. Finished - the job has finished speaking. When a second job finishes, this one will be deleted. You may click Restart to repeat the job. Note: Messages, Warnings, and Screen Reader Output do not appear in this list. See the Handbook for more information.
ML: പിശകില്‍ തുടരുക
Ratio: 0.011834319526627219

----

In [29]:
long_pairs = df.sort_values(
    "length_ratio",
    ascending=False
).head(20)

for _, row in long_pairs.iterrows():
    print("\n-------------------------")
    print("EN:", row["src"])
    print("ML:", row["tgt"])
    print("Ratio:", row["length_ratio"])


-------------------------
EN: Polygala (New World Clade) Muraltia
ML: Polygala (New World Clade) Muraltia അംബോറല്ലെസീ · അക്കാന്തേസീ · അക്കാനിയേസീ · അക്കാരിയേസീ · അക്കാറ്റോകാർപേസീ · അക്കോറേസീ · അക്വിഫോളിയേസീ · അർട്ടിക്കേസീ · അഡോക്സേസീ · അതെരോസ്പേർമറ്റേസീ · അന്നോനേസീ · അനാക്കാർഡിയേസീ · അനാക്യാമ്പ്സെറോറ്റേസീ · അനാർത്രിയേസീ · അനിസോഫൈല്ലേസീ · അപ്പിയേസീ · അപ്പോഡാന്തേസീ · അപ്പോനോജെറ്റോനേസീ · അപ്പോസൈനേസീ · അഫ്ലോയിയെസീ · അഫാനോപെറ്റാലേസീ · അമരാന്തേസീ · അമരൈല്ലിഡേസീ · അരക്കേസീ · അരക്കേസീ · അരാലിയേസീ · അരിസ്റ്റലോക്കിയേസീ · അലിസ്മറ്റേസീ · അസ്പരാഗേസീ · അസ്റ്റെറോപീയേസീ · അസ്റ്റേലിയേസീ · ആക്ടിനിഡിയേസീ · ആർഗോഫൈല്ലേസീ · ആൽട്ടിഞ്ചിയേസീ · ആസ്ട്രേസീ · ആസ്ട്രോബൈലിയേസീ · ആൽസ്യൂഓസ്മിയേസീ · ആൽസ്സ്ട്രോയെമെറിയേസീ · ആൻസിസ്റ്റ്രോക്ലാഡേസീ · ഇക്സിലിറിയേസീ · ഇക്സോണാന്തേസീ · ഇറിഡേസീ · ഇലാഗ്നേസീ · ഇലിയോകാർപേസീ · ഇർവിൻഗിയേസീ · ഈക്സ്റ്റോക്സികേസീ · ഉൾമേസീ · എംബ്ലിഞ്ജിയേസീ · എക്ഡൈയോകോളിയേസീ · എബെണേസീ · എരിത്രോസൈലേസീ · എറികേസീ · എറിയോകോളേസീ · എലാറ്റിനേസീ · എസ്ക്കലോണിയേസീ · ഐക്കാസിനേസീ · ഐറ്റേസീ · ഐസോയേസീ · ഒക്നേസീ · ഒനാഗ്ര

In [30]:
df["src_chars"] = df["src"].str.len()
df["tgt_chars"] = df["tgt"].str.len()

print(
    df[["src_chars", "tgt_chars"]].describe()
)

          src_chars     tgt_chars
count  1.481107e+06  1.481107e+06
mean   5.890681e+01  6.723218e+01
std    5.435774e+01  5.646733e+01
min    5.000000e+00  5.000000e+00
25%    2.600000e+01  3.300000e+01
50%    4.300000e+01  5.200000e+01
75%    7.400000e+01  8.300000e+01
max    2.019300e+04  6.100000e+03


In [31]:
short_src = df[df["src_words"] <= 3]

print("Source sentences with <=3 words:", len(short_src))

for _, row in short_src.head(20).iterrows():
    print("\nEN:", row["src"])
    print("ML:", row["tgt"])

Source sentences with <=3 words: 187429

EN: what has happened?
ML: എന്താണു സംഭവം?

EN: They didn't talk.
ML: അവര് മിണ്ടിയിട്ടില്ല.

EN: Well, anything!
ML: ആ, എന്തേലും ആവട്ട്‌ !

EN: I can go
ML: എനിക്ക് കഴിയും

EN: Parish priest Fr.
ML: മേരീസ് ഇടവകാംഗമായ ഫാ.

EN: But India won.
ML: പക്ഷേ, ഇന്ത്യ ജയിച്ചു.

EN: says Mayawati
ML: """ മായാവതി പറയുന്നു"

EN: What's up
ML: ങ്ങാ എന്താടി….

EN: Consider these.
ML: ശ്രദ്ധിക്കാം ഈ കാര്യങ്ങള്‍.

EN: Win for India
ML: ഇന്ത്യക്കായി നേടിയത്

EN: They shot him.
ML: അവർ അവനെ വെടിവെച്ചുകൊന്നു.

EN: Wonderful, wonderful technology
ML: അത്ഭുതകരമായ സാങ്കേതികവിദ്യ

EN: Dont forget that.
ML: എന്ന കാര്യം വിസ്മരിക്കരുത്.

EN: Trains partially cancelled
ML: പാസ്സഞ്ചര്‍ ട്രെയിനുകള്‍ ഭാഗികമായി റദ്ദു ചെയ്തു

EN: Chances of victory
ML: വിജയിക്കുന്നു സാധ്യത

EN: _Add Files
ML: ഫയലുകള്‍ ചേര്‍ക്കുക

EN: for release
ML: മോചനത്തിനായി

EN: Vintage cars
ML: വിന്റേജ് കാറുകൾ

EN: This contentious battle
ML: ഇക്കുറി പൊരിഞ്ഞ പോരാട്ടമാ .

EN: Not illegal.
ML: അല്ലാതെ നിയമവി

In [32]:
# Candidate quality filters

valid_basic = (
    df["src"].notna() &
    df["tgt"].notna() &
    (df["src"].str.strip() != "") &
    (df["tgt"].str.strip() != "")
)

no_web_noise = ~df["possible_web_noise"]

reasonable_length = (
    (df["src_words"] <= 100) &
    (df["tgt_words"] <= 100)
)

reasonable_ratio = (
    (df["length_ratio"] >= 0.2) &
    (df["length_ratio"] <= 3.0)
)

reasonable_malayalam = df["latin_ratio"] <= 0.50

candidate = (
    valid_basic &
    no_web_noise &
    reasonable_length &
    reasonable_ratio &
    reasonable_malayalam
)

print("Original:", len(df))
print("After candidate filtering:", candidate.sum())
print("Removed:", (~candidate).sum())
print("Retention:", round(candidate.mean() * 100, 2), "%")

Original: 1481107
After candidate filtering: 1471729
Removed: 9378
Retention: 99.37 %
